# Double DQN — Deep Reinforcement Learning with Double Q-learning (2016)

_Papers · Reinforcement Learning_

**Stop a Deep Q-Network from over-rating its own actions: pick the next action with one network, but score it with another.**

---

This notebook is a practice scaffold for the **Double DQN — Deep Reinforcement Learning with Double Q-learning (2016)** lesson. Run the cells, then tackle the practice problems at the bottom. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / scikit-learn / matplotlib ship with Colab.
!pip install -q gymnasium
import numpy as np, matplotlib.pyplot as plt

## Reference implementation — gymnasium + PyTorch (runs in Colab)

In [ ]:
# Double DQN vs vanilla DQN on CartPole-v1 — Colab:  !pip install gymnasium
import random, collections
import numpy as np, torch, torch.nn as nn, gymnasium as gym

# ---- 0. Recompute the WORKED EXAMPLE (must match the lesson: 4.168 vs 3.475) ----
R, gamma_ex = 1.0, 0.99
q_online_ex = torch.tensor([3.0, 4.0])     # online net Q(s', .)
q_target_ex = torch.tensor([3.2, 2.5])     # target net Q^-(s', .)
y_dqn  = R + gamma_ex * q_target_ex.max()                       # DQN: max over target net
a_star = q_online_ex.argmax()                                    # Double DQN: online SELECTS
y_ddqn = R + gamma_ex * q_target_ex[a_star]                     # ... target net EVALUATES
print(f"worked example  Y_DQN={y_dqn:.3f}  Y_DoubleDQN={y_ddqn:.3f}  gap={y_dqn-y_ddqn:.3f}")
assert abs(float(y_dqn) - 4.168) < 1e-3 and abs(float(y_ddqn) - 3.475) < 1e-3

# ---- 1. The DQN, with a DOUBLE switch on the target ----
DOUBLE = True            # <-- ABLATION: set False for vanilla DQN, watch overestimation return
env = gym.make("CartPole-v1")
n_obs, n_act = env.observation_space.shape[0], env.action_space.n
def make_q():
    return nn.Sequential(nn.Linear(n_obs,128), nn.ReLU(),
                         nn.Linear(128,128), nn.ReLU(), nn.Linear(128,n_act))
q      = make_q()
q_targ = make_q(); q_targ.load_state_dict(q.state_dict())
opt    = torch.optim.Adam(q.parameters(), lr=1e-3)
buf    = collections.deque(maxlen=50_000)
gamma, batch, sync = 0.99, 64, 500

def act(state, eps):
    if random.random() < eps: return env.action_space.sample()
    with torch.no_grad():
        return int(q(torch.as_tensor(state, dtype=torch.float32)).argmax())

def td_target(r, s2, d):
    with torch.no_grad():                                  # target is a constant: no grad
        if DOUBLE:
            a_sel = q(s2).argmax(dim=1, keepdim=True)       # SELECT with ONLINE net (theta)
            q_next = q_targ(s2).gather(1, a_sel).squeeze(1) # EVALUATE with TARGET net (theta-)
        else:
            q_next = q_targ(s2).max(dim=1).values           # vanilla DQN: max over target net
        return r + gamma * q_next * (1.0 - d)

def learn():
    if len(buf) < batch: return None
    s,a,r,s2,d = map(np.array, zip(*random.sample(buf, batch)))
    s  = torch.as_tensor(s,  dtype=torch.float32); s2 = torch.as_tensor(s2, dtype=torch.float32)
    a  = torch.as_tensor(a,  dtype=torch.int64);   r  = torch.as_tensor(r,  dtype=torch.float32)
    d  = torch.as_tensor(d,  dtype=torch.float32)
    q_sa = q(s).gather(1, a.unsqueeze(1)).squeeze(1)        # Q(s,a;theta)
    y    = td_target(r, s2, d)
    loss = nn.functional.mse_loss(q_sa, y)
    opt.zero_grad(); loss.backward(); opt.step()
    return float(q_sa.mean())                               # track the mean PREDICTED Q-value

steps = 0
for ep in range(300):
    state,_ = env.reset(); eps = max(0.02, 1.0 - ep/150); done=False; qsum=[]
    while not done:
        action = act(state, eps)
        nxt, rew, term, trunc, _ = env.step(action); done = term or trunc
        buf.append((state, action, rew, nxt, float(term))); state = nxt
        m = learn();  steps += 1
        if m is not None: qsum.append(m)
        if steps % sync == 0: q_targ.load_state_dict(q.state_dict())
    if ep % 50 == 0 and qsum:
        print(f"ep {ep:3d}  mean predicted Q = {np.mean(qsum):6.2f}   ({'Double' if DOUBLE else 'vanilla'} DQN)")
# vanilla DQN's mean predicted Q drifts ABOVE the true return (~the episode length discounted);
# Double DQN keeps it lower and closer to truth. Flip DOUBLE to compare.

## Visualize the data & results

_Does the single max really over-estimate, and does the double estimator fix it? Reproduce the paper's overestimation effect on a toy state whose true action-values are all 0._

In [ ]:
import numpy as np
# Reproduce Double DQN's core claim on a toy: a state with m actions, ALL true value 0.
# Estimates are unbiased Gaussian noise. Single max (DQN) over-estimates; double estimator does not.
rng = np.random.default_rng(0)
m, trials = 8, 4000
single, double = [], []
for sigma in [0.0, 0.5, 1.0, 1.5, 2.0]:
    s_vals, d_vals = [], []
    for _ in range(trials):
        qA = rng.normal(0, sigma, m)          # estimate set A (e.g. online net), true=0
        qB = rng.normal(0, sigma, m)          # estimate set B (e.g. target net), true=0, independent
        s_vals.append(qB.max())               # SINGLE max (DQN target operator)
        d_vals.append(qB[qA.argmax()])        # DOUBLE: select with A, evaluate with B
    single.append(round(float(np.mean(s_vals)), 3))
    double.append(round(float(np.mean(d_vals)), 3))
print("noise:  ", [0.0, 0.5, 1.0, 1.5, 2.0])
print("single: ", single)   # -> [0.0, 0.717, 1.429, 2.144, 2.86]   over-estimates
print("double: ", double)   # -> ~[0, 0, 0, 0, 0]                    unbiased

## Practice

Try each one in the empty cell below it, then reveal the worked solution.

**Problem 1.** One transition: $R_{t+1}=0$, $\gamma=0.9$, two next actions. Online net: $Q(s',a_0;\theta)=5,\ Q(s',a_1;\theta)=2$. Target net: $Q(s',a_0;\theta^-)=1,\ Q(s',a_1;\theta^-)=6$. Compute BOTH the DQN target and the Double DQN target, and explain the gap.

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- DQN: take $\max$ over the TARGET net. — _$\max(1,6)=6$, achieved at $a_1$. $Y^{DQN}=0+0.9\times 6 = 5.4$._
- Double DQN: SELECT with the online net. — _$\arg\max(5,2)=a_0$ &mdash; the online net prefers $a_0$._
- EVALUATE that $a_0$ in the target net. — _$Q(s',a_0;\theta^-)=1$, so $Y^{DoubleDQN}=0+0.9\times 1 = 0.9$._

**Answer:** $Y^{DQN}=5.4$, $Y^{DoubleDQN}=0.9$. DQN grabbed the target net's lucky $6$ (action $a_1$); Double DQN insisted on the ONLINE net's pick ($a_0$), which the target net rates at only $1$. The $4.5$ gap is the overestimation Double DQN removes here.

</details>

**Problem 2.** ABLATION. In the Double DQN target you replace $\arg\max_{a'}Q(s',a';\theta)$ with $\arg\max_{a'}Q(s',a';\theta^-)$ &mdash; selecting with the TARGET net instead of the online net. What target does this collapse to, and why?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Note that the outer evaluation already uses $\theta^-$. — _So now BOTH the selection and the evaluation use $\theta^-$._
- Recall $Q(s',\arg\max_{a'}Q(s',a';\theta^-);\theta^-) = \max_{a'}Q(s',a';\theta^-)$. — _Indexing a net at its own argmax just returns its max._

**Answer:** It collapses to the vanilla DQN target $Y^{DQN}=r+\gamma\max_{a'}Q(s',a';\theta^-)$. Selecting and evaluating with the SAME network is exactly the single-estimator $\max$ that over-estimates &mdash; the decoupling, and its benefit, are gone.

</details>

**Problem 3.** In the toy bias experiment, the TRUE value of all actions is $0$ and estimates are unbiased noise. Why does the single estimator's average come out ABOVE $0$ while the double estimator's stays at $0$?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Single estimator: $\max$ of several mean-zero noisy numbers. — _The max selects the largest positive error, so its expectation is $\gt 0$ &mdash; it grows with the noise (Theorem 1's $\sqrt{C/(m-1)}$)._
- Double estimator: pick the argmax with set A, read its value in INDEPENDENT set B. — _The action that had a lucky-large error in A has only an AVERAGE (mean-zero) value in B._

**Answer:** The single $\max$ couples selection and evaluation on the same noisy sample, so the lucky positive error is both chosen and reported &mdash; a positive bias. The double estimator evaluates on independent noise, so the bias cancels and the average stays at the true value $0$.

</details>